# Figure 5 — can it be displayed on phage

FoldIndex and the charge-hydropathy plane, plus the construct-format call from `src/express.py`.

The counter-intuitive read: **disorder is good**. A segment that is disordered in the intact receptor is one whose isolated peptide samples roughly the native ensemble, so a binder raised on it has a real shot on the cell. A segment predicted to fold will not reproduce that fold as a peptide.

## Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import style

style.apply()
if not style.available():
    print("Arial not found, matplotlib fell back to DejaVu Sans")

RESULTS, FIGS = ROOT / "results", ROOT / "figures"
FIGS.mkdir(exist_ok=True)
PAL = style.PALETTE
SEGMENTS = ["N-term", "ECL1", "ECL2", "ECL3"]

In [ ]:
def strip(ax, groups, values_by_group, log=False):
    """Jittered points with a median bar. Used in several panels."""
    for i, g in enumerate(groups):
        x = np.asarray(values_by_group[g], dtype=float)
        x = x[~np.isnan(x)]
        if not len(x):
            continue
        ax.scatter(np.random.normal(i, 0.09, len(x)), x, s=6, alpha=0.45,
                   color=PAL.get(g, "0.4"), edgecolors="none")
        ax.hlines(np.median(x), i - 0.28, i + 0.28, color="k", lw=1.6)
    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels(groups)
    if log:
        ax.set_yscale("log")

## Load

In [ ]:
dt = pd.read_csv(RESULTS / "display_tractability.csv")
seg = pd.read_csv(RESULTS / "segments_long.csv")
m = seg.merge(dt[["entry_name", "segment", "fold_index", "display_score",
                  "recommended_format"]], on=["entry_name", "segment"])

dt.groupby(["segment", "recommended_format"]).size().unstack(fill_value=0)

## Panels

In [ ]:
fig, ax = plt.subplot_mosaic([["ch", "fi"], ["fmt", "score"]],
                             figsize=(10.5, 7.8), constrained_layout=True)

a = ax["ch"]
H = (m["gravy"] + 4.5) / 9.0
R = m["charge_density"].abs()
for s in SEGMENTS:
    k = (m["segment"] == s).values
    a.scatter(H[k], R[k], s=10, alpha=0.55, color=PAL[s], label=s, edgecolors="none")
hh = np.linspace(float(H.min()), float(H.max()), 50)
a.plot(hh, 2.785 * hh - 1.151, "k--", lw=1)
a.annotate("FoldIndex = 0", (hh[-1], 2.785 * hh[-1] - 1.151), fontsize=7,
           ha="right", va="bottom", color="0.3")
a.set_xlabel("mean scaled hydrophobicity  <H>")
a.set_ylabel("mean net charge  |<R>|")
a.set_title("Charge-hydropathy: above line = disordered")
a.legend(fontsize=7)

a = ax["fi"]
for s in SEGMENTS:
    x = m[m.segment == s]
    a.scatter(x["length"], x["fold_index"], s=10, alpha=0.55, color=PAL[s],
              label=s, edgecolors="none")
a.axhline(0, color="0.5", lw=1, ls="--")
a.set_xscale("log")
a.set_xlabel("length (aa)")
a.set_ylabel("FoldIndex")
a.set_title("Below 0 = disordered = peptide is faithful")
a.legend(fontsize=7)

a = ax["fmt"]
ct = (dt.groupby(["segment", "recommended_format"]).size()
        .unstack(fill_value=0).reindex(SEGMENTS).fillna(0))
bottom = np.zeros(len(ct))
for col in ct.columns:
    a.bar(ct.index, ct[col].values, bottom=bottom,
          color=style.FORMAT_PALETTE.get(col, "0.6"), label=col)
    bottom += ct[col].values
a.set_ylabel("receptors")
a.set_title("Recommended construct format")
a.legend(fontsize=6.5)

strip(ax["score"], SEGMENTS,
      {s: dt.loc[dt.segment == s, "display_score"].values for s in SEGMENTS})
ax["score"].set_ylabel("display tractability score")
ax["score"].set_title("Composite (heuristic — see src/express.py)")

fig.savefig(FIGS / "05_displayability.png")